In [1]:
GB = 1e9

def full_ft_gb(n):
    "fp32 training: weight + gradient + Adam mean + Adam variance = 16 bytes/param."
    return n * 16 / GB

def lora_train_gb(n, trainable_frac=0.0018):
    "Frozen base in fp16; only the adapter carries gradients and optimizer state."
    base = n * 2 / GB
    adapter = n * trainable_frac * 16 / GB
    return base + adapter

print(f"{'model':14s}{'full FT (fp32)':>18s}{'LoRA':>14s}")

for name, n in [
    ("distilgpt2", 82e6),
    ("Llama-2-7B", 7e9),
    ("Llama-2-70B", 70e9),
]:
    print(
        f"{name:14s}"
        f"{full_ft_gb(n):15,.1f} GB"
        f"{lora_train_gb(n):11,.1f} GB"
    )

print("\nA single high-end GPU tops out around 80 GB.")
print(
    "Full fine-tuning a 7B model in fp32 (~112 GB) "
    "does not fit; LoRA (~14 GB) does."
)
print(
    "And each new task is a whole fresh 112 GB copy for full FT "
    "— but just a few MB adapter for LoRA."
)

model             full FT (fp32)          LoRA
distilgpt2                1.3 GB        0.2 GB
Llama-2-7B              112.0 GB       14.2 GB
Llama-2-70B           1,120.0 GB      142.0 GB

A single high-end GPU tops out around 80 GB.
Full fine-tuning a 7B model in fp32 (~112 GB) does not fit; LoRA (~14 GB) does.
And each new task is a whole fresh 112 GB copy for full FT — but just a few MB adapter for LoRA.


In [2]:
import numpy as np
rng = np.random.default_rng(0)

d = 768

# A rank-6 signal plus a little full-rank noise.
true_rank = 6
signal = (
    rng.standard_normal((d, true_rank))
    @ rng.standard_normal((true_rank, d))
)
dW = signal + 0.02 * rng.standard_normal((d, d))

# SVD measures how much energy lies in the top-r directions.
sv = np.linalg.svd(dW, compute_uv=False)
energy = np.cumsum(sv**2) / np.sum(sv**2)

print(f"a {d}x{d} update = {d*d:,} numbers if stored in full\n")
print(
    f"{'rank r':>7s}"
    f"{'% of update captured':>24s}"
    f"{'LoRA params (2·d·r)':>22s}"
)

for r in [1, 2, 4, 8, 16, 64, d]:
    params = 2 * d * r if r < d else d * d
    tag = "  <- full ΔW" if r == d else ""
    print(
        f"{r:7d}"
        f"{energy[r-1]*100:22.1f}%"
        f"{params:20,d}{tag}"
    )

r = 8
print(
    f"\nAt rank {r}: ~{energy[r-1]*100:.0f}% of the update, "
    f"for {2*d*r:,} params — "
    f"{d*d/(2*d*r):.0f}x smaller than the full {d*d:,}."
)

a 768x768 update = 589,824 numbers if stored in full

 rank r    % of update captured   LoRA params (2·d·r)
      1                  19.6%               1,536
      2                  38.5%               3,072
      4                  72.1%               6,144
      8                 100.0%              12,288
     16                 100.0%              24,576
     64                 100.0%              98,304
    768                 100.0%             589,824  <- full ΔW

At rank 8: ~100% of the update, for 12,288 params — 48x smaller than the full 589,824.


In [3]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """A frozen nn.Linear + a trainable low-rank detour B@A, scaled by alpha/r."""

    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base

        for p in self.base.parameters():
            p.requires_grad = False  # freeze W and bias

        d_in, d_out = base.in_features, base.out_features
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        detour = (self.dropout(x) @ self.A.T) @ self.B.T
        return self.base(x) + detour * self.scale


# Shape of GPT-2's fused Q/K/V projection: 768 -> 2304.
torch.manual_seed(0)
base = nn.Linear(768, 2304)
lora = LoRALinear(base, r=8, alpha=16)

x = torch.randn(4, 768)
print(
    "at init, LoRA(x) == base(x):",
    torch.allclose(lora(x), base(x)),
    " (because B=0, so B@A=0)"
)

trainable = sum(
    p.numel() for p in lora.parameters() if p.requires_grad
)
frozen = sum(
    p.numel() for p in lora.parameters() if not p.requires_grad
)
print(
    f"trainable (A,B): {trainable:,}"
    f"     frozen (W,bias): {frozen:,}"
    f"     -> {100*trainable/(trainable+frozen):.1f}% trainable"
)

# Once B moves away from zero, the detour changes the output.
with torch.no_grad():
    lora.B += 0.1 * torch.randn_like(lora.B)

print(
    "after nudging B, LoRA(x) == base(x):",
    torch.allclose(lora(x), base(x)),
    " (detour is now live)"
)

at init, LoRA(x) == base(x): True  (because B=0, so B@A=0)
trainable (A,B): 24,576     frozen (W,bias): 1,771,776     -> 1.4% trainable
after nudging B, LoRA(x) == base(x): False  (detour is now live)


In [4]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM
from transformers.pytorch_utils import Conv1D


class LoRALinear(nn.Module):
    """Frozen base layer + a trainable low-rank detour B@A.
    Handles nn.Linear and GPT-2's Conv1D.
    """

    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base

        for p in self.base.parameters():
            p.requires_grad = False

        if isinstance(base, Conv1D):
            d_in, d_out = base.weight.shape
        else:
            d_in, d_out = base.in_features, base.out_features

        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        detour = (self.dropout(x) @ self.A.T) @ self.B.T
        return self.base(x) + detour * self.scale


def add_lora(model, r=8, alpha=16, dropout=0.05):
    """Freeze the model, then wrap each fused Q/K/V projection."""
    for p in model.parameters():
        p.requires_grad = False

    for block in model.transformer.h:
        block.attn.c_attn = LoRALinear(
            block.attn.c_attn,
            r=r,
            alpha=alpha,
            dropout=dropout,
        )

    return model


torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

print(
    "what c_attn actually is:",
    type(model.transformer.h[0].attn.c_attn).__name__,
    "  weight",
    tuple(model.transformer.h[0].attn.c_attn.weight.shape),
    "\n",
)

ids = torch.tensor([[15496, 11, 995]])  # "Hello, world"

with torch.no_grad():
    before = model(ids).logits

add_lora(model, r=8, alpha=16)

with torch.no_grad():
    after = model(ids).logits

print(
    "identical to the base at injection:",
    torch.allclose(before, after, atol=1e-6),
    " (every B=0, so every detour is a no-op)\n",
)

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total = sum(p.numel() for p in model.parameters())

print(f"trainable : {trainable:>12,}")
print(f"frozen    : {total - trainable:>12,}")
print(
    f"-> {100 * trainable / total:.2f}% of the model trains "
    f"({total / trainable:,.0f}x fewer weights than full fine-tuning)\n"
)

print("everything the optimizer will see:")
for n, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {n:40s} {tuple(p.shape)}")

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1313.34it/s]


what c_attn actually is: Conv1D   weight (768, 2304) 

identical to the base at injection: True  (every B=0, so every detour is a no-op)

trainable :      147,456
frozen    :   81,912,576
-> 0.18% of the model trains (557x fewer weights than full fine-tuning)

everything the optimizer will see:
  transformer.h.0.attn.c_attn.A            (8, 768)
  transformer.h.0.attn.c_attn.B            (2304, 8)
  transformer.h.1.attn.c_attn.A            (8, 768)
  transformer.h.1.attn.c_attn.B            (2304, 8)
  transformer.h.2.attn.c_attn.A            (8, 768)
  transformer.h.2.attn.c_attn.B            (2304, 8)
  transformer.h.3.attn.c_attn.A            (8, 768)
  transformer.h.3.attn.c_attn.B            (2304, 8)
  transformer.h.4.attn.c_attn.A            (8, 768)
  transformer.h.4.attn.c_attn.B            (2304, 8)
  transformer.h.5.attn.c_attn.A            (8, 768)
  transformer.h.5.attn.c_attn.B            (2304, 8)


In [11]:
import os, random
from collections import Counter, defaultdict

import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from transformers.pytorch_utils import Conv1D

# ---- 1. Rebuild the task ----
ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [
    label
    for label, _ in Counter(ds["train"]["label"]).most_common(6)
]
topset = set(TOP)

def balanced_subset(split, per_class, seed):
    picked = defaultdict(list)

    for i, lab in enumerate(split["label"]):
        if lab in topset and len(picked[lab]) < per_class:
            picked[lab].append(i)

        if all(len(picked[label]) >= per_class for label in TOP):
            break

    flat = [i for label in TOP for i in picked[label]]
    random.Random(seed).shuffle(flat)
    return split.select(flat)

train = balanced_subset(ds["train"], per_class=100, seed=0)

tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])

# ---- 2. Freeze the base and wrap the attention ----
torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.resize_token_embeddings(len(tok), mean_resizing=False)
add_lora(model, r=8, alpha=16, dropout=0.05)

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total = sum(p.numel() for p in model.parameters())

print(
    f"trainable {trainable:,} / {total:,} "
    f"= {100*trainable/total:.2f}%\n",
    flush=True,
)

# ---- 3. Train only A and B ----
def encode(e):
    label_str = "<" + names[e["label"]] + ">"
    label_tid = tok.convert_tokens_to_ids(label_str)

    # Reserve one position for the label.
    prefix = tok(
        e["text"].strip() + " ",
        truncation=True,
        max_length=95,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = prefix + [label_tid]
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prefix) + [label_tid]

    padding = 96 - len(input_ids)
    input_ids += [tok.pad_token_id] * padding
    attention_mask += [0] * padding
    labels += [-100] * padding

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized = train.map(
    encode,
    remove_columns=train.column_names,
)
tokenized.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"],
)

args = TrainingArguments(
    output_dir="lora-out",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_steps=50,
    report_to="none",
    save_strategy="no",
    disable_tqdm=True,
)

Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=default_data_collator,
).train()

# ---- 4. Save the adapter and label embedding rows ----
state = {
    k: v
    for k, v in model.state_dict().items()
    if k.endswith(".A") or k.endswith(".B")
}
state["label_embeddings"] = (
    model.transformer.wte.weight.data[-len(TOP):].clone()
)

torch.save(state, "lora-adapter.pt")

print(
    f"\nsaved lora-adapter.pt — "
    f"{os.path.getsize('lora-adapter.pt'):,} bytes "
    f"({len(state)-1} adapter tensors + "
    f"the {len(TOP)} label rows)"
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1524.19it/s]


trainable 147,456 / 82,064,640 = 0.18%

{'loss': '2.62', 'grad_norm': '2.311', 'learning_rate': '0.0008911', 'epoch': '0.6667'}
{'loss': '1.328', 'grad_norm': '3.159', 'learning_rate': '0.00078', 'epoch': '1.333'}
{'loss': '0.7767', 'grad_norm': '6.51', 'learning_rate': '0.0006689', 'epoch': '2'}
{'loss': '0.3761', 'grad_norm': '1.972', 'learning_rate': '0.0005578', 'epoch': '2.667'}
{'loss': '0.2345', 'grad_norm': '6.76', 'learning_rate': '0.0004467', 'epoch': '3.333'}
{'loss': '0.1133', 'grad_norm': '0.3876', 'learning_rate': '0.0003356', 'epoch': '4'}
{'loss': '0.1039', 'grad_norm': '0.3016', 'learning_rate': '0.0002244', 'epoch': '4.667'}
{'loss': '0.08359', 'grad_norm': '0.3915', 'learning_rate': '0.0001133', 'epoch': '5.333'}
{'loss': '0.06011', 'grad_norm': '0.1847', 'learning_rate': '2.222e-06', 'epoch': '6'}
{'train_runtime': '2888', 'train_samples_per_second': '1.247', 'train_steps_per_second': '0.156', 'train_loss': '0.6329', 'epoch': '6'}

saved lora-adapter.pt — 613,339 by

In [10]:
print(
    "Supervised examples:",
    sum(
        bool((example["labels"] != -100).any())
        for example in tokenized
    ),
    "/",
    len(tokenized),
)

Supervised examples: 600 / 600


In [12]:
import os
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [
    label
    for label, _ in Counter(ds["train"]["label"]).most_common(6)
]

# A fresh pretrained distilgpt2.
tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])

model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.resize_token_embeddings(len(tok))
add_lora(model)

state = torch.load("lora-adapter.pt")
rows = state.pop("label_embeddings")

with torch.no_grad():
    model.transformer.wte.weight[-len(TOP):] = rows

missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

print(
    f"loaded {len(state)} adapter tensors + "
    f"{tuple(rows.shape)} label rows "
    f"from a {os.path.getsize('lora-adapter.pt'):,}-byte file"
)
print(
    f"unexpected keys: {len(unexpected)} "
    f" (the {len(missing):,} 'missing' keys are the frozen base)\n"
)

label_id_to_name = {
    tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i]
    for i in TOP
}
label_ids = list(label_id_to_name)

@torch.no_grad()
def classify(clause):
    "One forward pass; argmax restricted to the six label tokens."
    logits = model(
        **tok(clause.strip() + " ", return_tensors="pt")
    ).logits[0, -1]

    return label_id_to_name[
        max(label_ids, key=lambda t: logits[t].item())
    ]

val = (
    ds["validation"]
    .filter(lambda e: e["label"] in set(TOP))
    .shuffle(seed=2)
    .select(range(120))
)

preds = [classify(e["text"]) for e in val]
trues = [names[e["label"]] for e in val]
acc = sum(p == t for p, t in zip(preds, trues)) / len(val)

print(
    f"{'':18s}{'accuracy':>10s}"
    f"{'trainable':>14s}{'shipped per task':>19s}"
)
print(
    f"{'full fine-tune':18s}{'95.8%':>10s}"
    f"{'81,912,576':>14s}{'327 MB':>19s}"
)
print(
    f"{'LoRA r=8':18s}{acc:>9.1%}"
    f"{'147,456':>14s}"
    f"{os.path.getsize('lora-adapter.pt')/1e3:>16,.0f} KB"
)
print(f"\n(6 clause types, so random guessing = {1/len(TOP):.1%})")

print("\nwhere it still slips:")
shown = 0

for p, t, e in zip(preds, trues, val):
    if p != t and shown < 5:
        print(
            f"  true={t:20s} pred={p:20s} "
            f":: {e['text'][:52]}..."
        )
        shown += 1

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1232.91it/s]


loaded 12 adapter tensors + (6, 768) label rows from a 613,339-byte file
unexpected keys: 0  (the 77 'missing' keys are the frozen base)

                    accuracy     trainable   shipped per task
full fine-tune         95.8%    81,912,576             327 MB
LoRA r=8              99.2%       147,456             613 KB

(6 clause types, so random guessing = 16.7%)

where it still slips:
  true=Counterparts         pred=Notices              :: Administrative Agent shall have received from the Le...


In [7]:
supervised = sum(
    bool((example["labels"] != -100).any())
    for example in tokenized
)

print("Training examples:", len(tokenized))
print("Examples containing a supervised label:", supervised)

saved = torch.load("lora-adapter.pt", map_location="cpu")

for name, tensor in saved.items():
    if name.endswith(".B"):
        print(name, "norm:", tensor.norm().item())

Training examples: 600
Examples containing a supervised label: 339
transformer.h.0.attn.c_attn.B norm: 2.2037787437438965
transformer.h.1.attn.c_attn.B norm: 2.067504644393921
transformer.h.2.attn.c_attn.B norm: 2.0673770904541016
transformer.h.3.attn.c_attn.B norm: 2.1981759071350098
transformer.h.4.attn.c_attn.B norm: 2.597841501235962
transformer.h.5.attn.c_attn.B norm: 3.000159978866577


In [8]:
import torch
import torch.nn.functional as F

saved = torch.load("lora-adapter.pt", map_location="cpu")
rows = saved["label_embeddings"].float()

similarities = F.normalize(rows, dim=1) @ F.normalize(rows, dim=1).T
off_diagonal = ~torch.eye(len(rows), dtype=torch.bool)

print("Label-row norms:", rows.norm(dim=1).tolist())
print(
    "Cosine similarity between different label rows:",
    similarities[off_diagonal].min().item(),
    "to",
    similarities[off_diagonal].max().item(),
)
print(
    "Largest absolute difference from the first row:",
    (rows[1:] - rows[0]).abs().max().item(),
)

Label-row norms: [2.0225188732147217, 2.0225305557250977, 2.022526502609253, 2.022534132003784, 2.02254056930542, 2.0225586891174316]
Cosine similarity between different label rows: 0.9999998211860657 to 1.000000238418579
Largest absolute difference from the first row: 2.0779669284820557e-05


In [13]:
import os, random, time
from collections import Counter, defaultdict

import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from transformers.pytorch_utils import Conv1D


class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base

        for p in self.base.parameters():
            p.requires_grad = False

        if isinstance(base, Conv1D):
            d_in, d_out = base.weight.shape
        else:
            d_in, d_out = base.in_features, base.out_features

        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (
            (self.dropout(x) @ self.A.T) @ self.B.T
        ) * self.scale

    def merged(self):
        """Fold the adapter into the base weight."""
        with torch.no_grad():
            delta = (self.B @ self.A) * self.scale
            if isinstance(self.base, Conv1D):
                self.base.weight += delta.T
            else:
                self.base.weight += delta
        return self.base


def add_lora(model, r=8, alpha=16, dropout=0.05):
    for p in model.parameters():
        p.requires_grad = False

    for block in model.transformer.h:
        block.attn.c_attn = LoRALinear(
            block.attn.c_attn,
            r=r,
            alpha=alpha,
            dropout=dropout,
        )

    return model


ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [
    label
    for label, _ in Counter(ds["train"]["label"]).most_common(6)
]
topset = set(TOP)

tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])


def balanced_subset(split, per_class, seed):
    picked = defaultdict(list)

    for i, lab in enumerate(split["label"]):
        if lab in topset and len(picked[lab]) < per_class:
            picked[lab].append(i)

        if all(len(picked[label]) >= per_class for label in TOP):
            break

    flat = [i for label in TOP for i in picked[label]]
    random.Random(seed).shuffle(flat)
    return split.select(flat)


voice_train = balanced_subset(
    ds["train"], per_class=100, seed=7
)

torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.resize_token_embeddings(len(tok), mean_resizing=False)
add_lora(model, r=8, alpha=16, dropout=0.05)


def encode_lm(e):
    """Predict every real token; ignore padding."""
    enc = tok(
        e["text"].strip(),
        truncation=True,
        padding="max_length",
        max_length=96,
    )
    enc["labels"] = [
        token if mask == 1 else -100
        for token, mask in zip(
            enc["input_ids"], enc["attention_mask"]
        )
    ]
    return enc


voice_tok = voice_train.map(
    encode_lm,
    remove_columns=voice_train.column_names,
)
voice_tok.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"],
)

args = TrainingArguments(
    output_dir="voice-out",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_steps=75,
    report_to="none",
    save_strategy="no",
    disable_tqdm=True,
)

Trainer(
    model=model,
    args=args,
    train_dataset=voice_tok,
    data_collator=default_data_collator,
).train()

voice = {
    k: v
    for k, v in model.state_dict().items()
    if k.endswith(".A") or k.endswith(".B")
}
torch.save(voice, "lora-voice.pt")

print(
    f"\nsaved lora-voice.pt — "
    f"{os.path.getsize('lora-voice.pt'):,} bytes "
    f"({len(voice)} tensors)"
)

Map: 100%|██████████| 600/600 [00:00<00:00, 728.47 examples/s]


{'loss': '9.812', 'grad_norm': '0.5245', 'learning_rate': '0.0006711', 'epoch': '1'}
{'loss': '3.294', 'grad_norm': '0.5557', 'learning_rate': '0.0003378', 'epoch': '2'}
{'loss': '3.186', 'grad_norm': '0.91', 'learning_rate': '4.444e-06', 'epoch': '3'}
{'train_runtime': '1533', 'train_samples_per_second': '1.174', 'train_steps_per_second': '0.147', 'train_loss': '5.431', 'epoch': '3'}

saved lora-voice.pt — 594,613 bytes (12 tensors)


In [14]:
def load_adapter(model, path, n_rows=6):
    """Load adapter matrices and label rows, when present."""
    state = torch.load(path)
    rows = state.pop("label_embeddings", None)

    if rows is not None:
        with torch.no_grad():
            model.transformer.wte.weight[-n_rows:] = rows

    model.load_state_dict(state, strict=False)
    return model.eval()


PROMPT = "The parties hereby agree that"
CLAUSE = (
    "This Agreement shall be governed by and construed "
    "in accordance with the laws of the State of New York."
)

label_id_to_name = {
    tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i]
    for i in TOP
}
label_ids = list(label_id_to_name)


@torch.no_grad()
def generate(m, t, prompt=PROMPT, n=22):
    out = m.generate(
        **t(prompt, return_tensors="pt"),
        max_new_tokens=n,
        do_sample=False,
        pad_token_id=t.eos_token_id,
    )
    return t.decode(
        out[0], skip_special_tokens=True
    ).replace("\n", " ")


@torch.no_grad()
def classify(m, clause=CLAUSE):
    logits = m(
        **tok(clause.strip() + " ", return_tensors="pt")
    ).logits[0, -1]

    return label_id_to_name[
        max(label_ids, key=lambda t: logits[t].item())
    ]


# ---- Compare the plain base and two adapters ----
plain_tok = AutoTokenizer.from_pretrained("distilgpt2")
plain = AutoModelForCausalLM.from_pretrained(
    "distilgpt2"
).eval()

print("base, no adapter :", generate(plain, plain_tok)[:120])

base = AutoModelForCausalLM.from_pretrained("distilgpt2")
base.resize_token_embeddings(len(tok), mean_resizing=False)
add_lora(base)

load_adapter(base, "lora-voice.pt")
print("+ voice adapter  :", generate(base, tok)[:120])

load_adapter(base, "lora-adapter.pt")
print(
    "+ clause adapter :",
    f"tags the New York clause as {classify(base)!r}",
)
print(
    "                   "
    "(the adapted base object is reused for both adapters)\n"
)

# ---- Merge the active clause adapter into W ----
ids = tok(CLAUSE + " ", return_tensors="pt")

with torch.no_grad():
    before = base(**ids).logits

for blk in base.transformer.h:
    blk.attn.c_attn = blk.attn.c_attn.merged()

with torch.no_grad():
    after = base(**ids).logits

print("after merging into W:")
print(
    "  c_attn is a plain",
    type(base.transformer.h[0].attn.c_attn).__name__,
    "again — no wrapper to ship",
)
print(f"  same prediction: {classify(base)!r}")
print(
    "  logits agree to float32 round-off: "
    f"max |diff| = {(before - after).abs().max():.1e} "
    f"on logits spanning {before.min():.0f}..{before.max():.0f}\n"
)

d = base.config.n_embd
d_out = 3 * d
r = 8
mac_base = d * d_out
mac_detour = d * r + r * d_out

print("inference cost, per token, per layer:")
print(
    f"  base c_attn : {mac_base:>10,} multiply-adds"
)
print(
    f"  LoRA detour : {mac_detour:>10,} "
    f"(+{100*mac_detour/mac_base:.2f}%)"
)
print(
    f"  merged      : {mac_base:>10,} "
    "(+0.00%) — the detour is folded into W"
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 627.87it/s]


base, no adapter : The parties hereby agree that the parties agree to the terms of the Agreement.            


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 556.94it/s]


+ voice adapter  : The parties hereby agree that the terms and conditions of this Agreement shall be governed by the laws of the State of C
+ clause adapter : tags the New York clause as 'Governing Laws'
                   (the adapted base object is reused for both adapters)

after merging into W:
  c_attn is a plain Conv1D again — no wrapper to ship
  same prediction: 'Governing Laws'
  logits agree to float32 round-off: max |diff| = 1.1e-04 on logits spanning -153..8

inference cost, per token, per layer:
  base c_attn :  1,769,472 multiply-adds
  LoRA detour :     24,576 (+1.39%)
  merged      :  1,769,472 (+0.00%) — the detour is folded into W


In [15]:
from collections import Counter

import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.pytorch_utils import Conv1D


def quantize_4bit(w, block=64):
    """Block-wise absmax rounding to approximately 4-bit precision."""
    flat = w.detach().flatten().float()
    pad = (-flat.numel()) % block

    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])

    blocks = flat.view(-1, block)
    scale = blocks.abs().amax(
        dim=1, keepdim=True
    ).clamp(min=1e-8)

    q = torch.round(
        blocks / scale * 7
    ).clamp(-8, 7).to(torch.int8)

    return q, scale, pad, w.shape


def dequantize(q, scale, pad, shape):
    out = (q.float() / 7 * scale).flatten()

    if pad:
        out = out[:out.numel() - pad]

    return out.view(shape)


ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [
    label
    for label, _ in Counter(ds["train"]["label"]).most_common(6)
]

tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])

label_id_to_name = {
    tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i]
    for i in TOP
}
label_ids = list(label_id_to_name)

val = (
    ds["validation"]
    .filter(lambda e: e["label"] in set(TOP))
    .shuffle(seed=2)
    .select(range(120))
)
trues = [names[e["label"]] for e in val]


def build_tagger():
    m = AutoModelForCausalLM.from_pretrained("distilgpt2")
    m.resize_token_embeddings(len(tok), mean_resizing=False)
    add_lora(m)

    state = torch.load("lora-adapter.pt")
    rows = state.pop("label_embeddings")

    with torch.no_grad():
        m.transformer.wte.weight[-len(TOP):] = rows

    m.load_state_dict(state, strict=False)
    return m.eval()


@torch.no_grad()
def accuracy(m):
    def cls(clause):
        logits = m(
            **tok(clause.strip() + " ", return_tensors="pt")
        ).logits[0, -1]

        return label_id_to_name[
            max(label_ids, key=lambda t: logits[t].item())
        ]

    return sum(
        cls(e["text"]) == truth
        for e, truth in zip(val, trues)
    ) / len(val)


m = build_tagger()
print(f"fp32 frozen base + adapter : {accuracy(m):.1%}")

n_quant, err = 0, []

with torch.no_grad():
    for mod in m.modules():
        if isinstance(mod, (Conv1D, nn.Linear)):
            w = mod.weight.data
            dq = dequantize(*quantize_4bit(w))

            err.append(
                (w - dq).abs().mean().item()
                / w.abs().mean().item()
            )
            mod.weight.data = dq
            n_quant += w.numel()

print(
    f"quantized {n_quant:,} frozen weights in {len(err)} layers "
    f"(mean relative error {100*sum(err)/len(err):.1f}%)"
)
print(f"4-bit frozen base + adapter: {accuracy(m):.1%}")

P = 70e9
print("\nmemory to FINE-TUNE, by method (70B model):")
print(
    f"  full fine-tune, fp32       : {P*16/1e9:>7,.0f} GB"
    "   weights+grad+2 Adam states"
)
print(
    f"  LoRA, fp16 frozen base     : "
    f"{(P*2 + P*0.0018*16)/1e9:>7,.0f} GB"
    "   base resident, adapter trains"
)
print(
    f"  QLoRA, 4-bit frozen base   : "
    f"{(P*0.5 + P*0.0018*16)/1e9:>7,.0f} GB"
    "   base quantized to ~4 bits/param"
)
print(f"  {'a single 80 GB GPU':27s}: {80:>7,.0f} GB")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 972.36it/s]


fp32 frozen base + adapter : 99.2%
quantized 81,069,312 frozen weights in 25 layers (mean relative error 13.1%)
4-bit frozen base + adapter: 92.5%

memory to FINE-TUNE, by method (70B model):
  full fine-tune, fp32       :   1,120 GB   weights+grad+2 Adam states
  LoRA, fp16 frozen base     :     142 GB   base resident, adapter trains
  QLoRA, 4-bit frozen base   :      37 GB   base quantized to ~4 bits/param
  a single 80 GB GPU         :      80 GB
